# EverFlavor AI - Climate and Season: Carbon Footprint and What Is in Season

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: partly built.** Section 3b (meat: supply season and pasture season) is built, tested and runs
> without keys. Sections 2, 3 and 4 are still a plan: their functions are stubs (`NotImplementedError`) and
> are not called. "Run all" is safe; the first run downloads about 10 minutes of weather (section 3b).

---

## Purpose

Two more things a person may care about when choosing a dish:

- **Climate:** roughly how much greenhouse gas the ingredients cause (kg CO2-equivalent), so the Chef Agent
  can mention a lower-impact swap (lentils for beef) when someone asks.
- **Season:** which fruits and vegetables are in season this month where the person lives, so suggestions
  favor fresh, cheaper produce.

**Honesty rules:** without ingredient amounts, a dish gets an **impact class** from its main ingredients
(for example "contains beef: high impact"), never a precise kg CO2e per serving. Footprints are global or
French averages, not the person's farm or store. Seasons vary by region and weather: "usually in season".

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | `data/interim/recipes_all.parquet` (ingredient lists) | Notebook 01 (section 5.11) |
| Input | Our World in Data: GHG emissions per kg of food (Poore & Nemecek 2018), CC BY | Section 2, small CSV download |
| Input | AGRIBALYSE 3.2 (ADEME, France): 2,516 foods, Etalab Open License | Section 2, CSV download |
| Input | A seasonal produce calendar by month and region (USDA SNAP-Ed guide; license to confirm) | Section 3 |
| Output | `data/processed/ingredient_footprint.csv` (kg CO2e per kg, source, match) | Notebook 08 (section 2) |
| Output | `data/processed/recipe_impact.parquet` (impact class per recipe) | Notebook 08 (section 4) |
| Output | `data/processed/produce_seasons.csv` (item, region, months) | Notebook 08 (section 3) |
| Input | USDA ERS livestock data (monthly slaughter by species, 1944 on) | Section 3b, 1 MB zip, no key |
| Input | Open-Meteo past weather (daily, 2015-2024) for each grazing region | Section 3b, no key |
| Output | `data/processed/meat_supply_seasons.csv` (peak and low months per meat, US) | Section 3b |
| Output | `data/processed/pasture_seasons.csv` (grass-growing and finishing months by region) | Section 3b |

## Contents

1. Setup
2. Carbon Footprint per Ingredient
3. What Is in Season
4. Impact Class per Recipe
5. Team Decisions and Next Steps

---

## 1. Setup

The same setup as the other notebooks: it works unchanged in Google Colab, VS Code and Antigravity.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.util
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
IN_COLAB = importlib.util.find_spec("google.colab") is not None   # only exists in Colab


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "agent_tools", "calories", "charts", "checks", "cooking", "crew",
                      "cuisine", "diets", "environment", "features", "flags", "freshness", "ingredients",
                      "knowledge", "llm", "nutrition", "nutrition_quality", "parsing", "pipeline",
                      "progress", "recommend", "reporting", "review", "safety", "seasons", "sources",
                      "stores", "variants"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
FOOTPRINT_DIR = Path("data/raw/footprint")
for folder in [FOOTPRINT_DIR, Path("data/processed")]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

Runtime        : Local Jupyter (VS Code / Antigravity / JupyterLab)
Python         : 3.12.10
Project folder : ~\Downloads\jupiter-exploration\The-EverFlavor-AI
Notebook 01 outputs found.


---

## 2. Carbon Footprint per Ingredient

| Source | What it has | License | Use |
|---|---|---|---|
| [Our World in Data](https://ourworldindata.org/grapher/ghg-per-kg-poore) (Poore & Nemecek 2018, *Science*) | Global average kg CO2e per kg for about 40 foods (beef, lamb, cheese, rice, lentils ...) from ~38,700 farms in 119 countries | CC BY ("completely open access under the Creative Commons BY license") | Main source: broad food groups, worldwide |
| [AGRIBALYSE 3.2](https://doc.agribalyse.fr/documentation-en/agribalyse-data/data-access) (ADEME) | 2,516 finished foods sold in France, climate change per kg and by life-cycle stage | Etalab Open License (reuse, also commercial, with the source and date credited) | Finer detail for prepared foods; French averages |

**Matching:** each of our ingredient names is matched to a food group ("ground beef" -> Beef (beef herd),
"cheddar cheese" -> Cheese), like the USDA matches of notebook 01, with a hand check of the top matches.

In [3]:
OWID_GHG_URL = "https://ourworldindata.org/grapher/ghg-per-kg-poore.csv?v=1&csvType=full&useColumnShortNames=false"
AGRIBALYSE_PAGE = "https://doc.agribalyse.fr/documentation-en/agribalyse-data/data-access"


def footprint_table(folder: Path, refresh: bool = False) -> pd.DataFrame:
    """Download Our World in Data's GHG per kg table (cached): food, kg_co2e_per_kg, source."""
    raise NotImplementedError("sketch: footprint download")


def match_footprints(ingredients: list[str], footprints: pd.DataFrame) -> pd.DataFrame:
    """Match ingredient names to footprint foods: ingredient, food, kg_co2e_per_kg, match (exact / group / hand-check)."""
    raise NotImplementedError("sketch: footprint matching")

---

## 3. What Is in Season

Sources checked 2026-10-09 (license read; "tested" means a real download worked without a key).

| For | Source | What it gives | License | Access |
|---|---|---|---|---|
| Fruit and vegetables | [Foodland Ontario Availability Guide](https://data.ontario.ca/dataset/foodland-ontario-fruits-and-vegetables-availability-guide) | Month-by-month availability (0 / 1) for local fruit and vegetables | Open Government Licence - Ontario | XLSX, tested (20 fruits x 12 months) |
| Crops worldwide | [GEOGLAM Crop Monitor sub-national crop calendars v1.3](https://zenodo.org/records/15850408) | Planting to harvest by region: wheat, maize, rice, soybeans, sorghum, millet, beans, teff | CC BY 4.0 | Zenodo zip (550 MB) |
| Past harvests, US | [USDA NASS Quick Stats](https://quickstats.nass.usda.gov/api) Crop Progress | Weekly % planted / harvested by crop and state, many past years | US government work | API, free key needed (not tested) |
| Meat, eggs, milk, US | USDA NASS Quick Stats (Livestock Slaughter, Chickens and Eggs, Milk Production) | Monthly head counts and production by species: when supply peaks (lamb in spring ...) | US government work | Same API and key |
| Seafood, EU | [EUMOFA](https://eumofa.eu) monthly first sales | Volume and price by species, month and country: when catches peak | To check | Bulk download (not tested) |
| Seafood, US | [NOAA FOSS](https://www.fisheries.noaa.gov/foss) commercial landings | Pounds and value by species and state: yearly through the API (tested); monthly only through the web tool's CSV export | US government work | API tested (yearly) |
| Seafood availability | [NOAA Fisheries species pages](https://www.fisheries.noaa.gov/species-directory) (FishWatch's profiles moved here) | One line per US species: "Year-round, with peaks in the summer" (brown shrimp) | US government work; crawling allowed (robots.txt) | Web pages, tested (red snapper, brown shrimp, sea scallop); read politely, cached |
| Seafood rules | State and federal closed seasons (for example Texas Gulf shrimp: closed May 15 to about July 15) | When a catch is allowed | Public notices, no dataset | By hand, cited per row |
| Past weather | [Open-Meteo historical API](https://open-meteo.com/en/docs/historical-weather-api) (ERA5) | Daily temperature and rain since 1940 for any place | CC BY 4.0, free for non-commercial use | API, no key, tested (Houston 2025) |

Not usable: NOAA FishWatch's API is retired (every address now redirects to NOAA's seafood page; it never used
keys, so no key brings it back), but its availability text lives on in the species pages above; the USDA SNAP-Ed and state
harvest calendars are PDFs without a stated license.

**Honesty rules:** livestock is raised all year, so for meat "in season" means "supply usually peaks"; for
wild seafood it means "the season is open and landings are high". Ontario months are a proxy for northern
US states, not for Houston: Texas months need a Texas source or the NASS harvest weeks.

**Planned tables:** `produce_seasons.csv` (`item`, `region`, `months`, `source`) and
`seafood_meat_seasons.csv` (`item`, `region`, `peak_months`, `basis`, `source`).

In [4]:
def produce_seasons(region: str) -> pd.DataFrame:
    """Months each fruit and vegetable is usually in season in `region`."""
    raise NotImplementedError("sketch: seasonal calendar")

---

## 3b. Meat: Supply Season and Pasture Season (two different things)

Farm animals are raised all year, so meat has no harvest season. Two separate ideas are kept apart, and neither
is about freshness (fresh or spoiled is notebook 05's photo check; how long meat keeps is notebook 04's shelf life).
The code is in `src/everflavor/seasons.py`; nothing here needs a key.

| Idea | Question it answers | Data | Wording for the agent |
|---|---|---|---|
| **Supply season** | When is this meat most plentiful (and often cheaper)? | [USDA ERS](https://www.ers.usda.gov/data-products/livestock-and-meat-domestic-data) monthly slaughter by species, 1944 on (US government work) | "lamb is most plentiful in March and April" |
| **Pasture season** | When do animals in this region usually graze fresh, growing grass? | Past weather, 2015-2024, from [Open-Meteo](https://open-meteo.com) (ERA5, CC BY 4.0) | "grass-fed meat from this region is often finished on fresh pasture in these months" |

**Why pasture season matters:** grazing on fresh grass raises omega-3 fats and antioxidants in beef and lamb
compared with grain feeding, and the finishing season changes the fat profile too (autumn-finished UK beef had
more omega-3 and CLA; spring-finished beef a lower omega-6 / omega-3 ratio). The evidence is real but mixed, and
the farm's own feeding matters more than the calendar, so the agent says "often", never "better".

**Supply logic:** slaughter per working day (plants run Monday-Friday, so short months and five-weekend months do
not count as dips), divided by that year's average (growth does not count), median over the last 10 complete
years (odd years such as 2020 do not count). A **peak** month is 5% or more above average in the median year
and above average in most years; a **low** month mirrors it; a meat whose months differ by less than 10% is
**steady all year**. The numbers are for the US; other countries need their own statistics.

**Pasture logic:**

- **Between the tropics** (23.44 deg N to 23.44 deg S) there is **no pasture season**: quality depends on each
  farm's pasture and feed, which we do not rate (no companies yet). No weather is downloaded for these places.
- Elsewhere each month of each year gets **temperature factor x moisture factor** (0 to 1 each). Temperature: 0 at
  a monthly mean of 4 deg C or less, 1 at 9 deg C or more. Moisture: rain against the water plants lose (FAO
  reference evapotranspiration) over **that month and the one before**, since soil stores water: 0 at 0.15 or less
  (desert), 1 at 0.5 or more. The usual score is the mean over the years; 0.5 or more is a **grass month**. The
  Southern Hemisphere needs no special rule: its weather already has its seasons.
- Green spells shorter than **3 months** in a row are not a season (desert grass after winter rain); 11 or more
  grass months is **year-round grass** (quality depends on the farm, as in the tropics).
- **Finishing months** are grass months that follow a grass month: meat reflects the last weeks of diet, so the
  animal must have grazed fresh grass for a while.
- Only for grazing animals (beef, veal, lamb, mutton, goat, bison, venison); pork and chicken are mostly fed indoors.

The four limits were checked against known grazing months in 10 regions (the check below): 4 months off in 120,
against 17 for the first guess (5-10 deg C, 0.3-0.7). **Known limits:** one weather point per region; irrigated
and sown winter pastures are not modeled; Mongolia's season comes out about a month late.

**Outputs:** `data/processed/meat_supply_seasons.csv` and `data/processed/pasture_seasons.csv`.
The first run downloads 10 years of daily weather per region: Open-Meteo allows about one region a minute,
so it takes about 10 minutes once; later runs read the saved copies in `data/raw/weather/`.

In [5]:
from everflavor import seasons

MONTH_NAMES = seasons.MONTHS
ERS_DIR = Path("data/raw/ers_livestock")
WEATHER_DIR = Path("data/raw/weather")

ers_zip = seasons.download_ers_livestock(ERS_DIR, refresh=REFRESH_DOWNLOADS)
slaughter = seasons.read_ers_slaughter(ers_zip)
supply_index = seasons.supply_index(slaughter)
meat_supply = seasons.supply_seasons(supply_index)
meat_supply.to_csv("data/processed/meat_supply_seasons.csv", index=False)

print(f"USDA ERS monthly slaughter: {slaughter['year'].min()}-{slaughter['year'].max()}; "
      f"last {seasons.SUPPLY_YEARS} complete years used\n")
table = supply_index.pivot(index="meat", columns="month", values="index").round(2)
table.columns = list(MONTH_NAMES)
print("Each month against its year's average (1.00 = average):")
print(table.to_string())
print()
for row in meat_supply.itertuples():
    peaks = ", ".join(MONTH_NAMES[m - 1] for m in row.peak_months) or "-"
    lows = ", ".join(MONTH_NAMES[m - 1] for m in row.low_months) or "-"
    print(f"{row.meat:8} {row.pattern:16} peaks: {peaks:16} lows: {lows}")

USDA ERS monthly slaughter: 1944-2026; last 10 complete years used

Each month against its year's average (1.00 = average):
          Jan   Feb   Mar   Apr   May   Jun   Jul   Aug   Sep   Oct   Nov   Dec
meat                                                                           
beef     0.98  0.99  0.99  1.00  0.98  1.05  1.00  1.02  1.00  1.03  1.01  0.97
chicken  0.99  0.99  0.98  0.99  0.99  1.02  1.00  1.03  1.02  1.04  0.96  0.96
lamb     0.91  0.96  1.06  1.07  0.98  1.00  0.95  0.98  0.99  1.01  1.02  1.05
mutton   0.83  0.88  0.90  1.10  1.04  1.05  1.02  1.09  1.07  1.01  0.97  1.03
pork     1.02  1.02  1.01  0.99  0.93  0.97  0.92  0.99  1.01  1.07  1.04  1.03
turkey   0.96  0.98  0.98  0.97  0.98  1.04  0.99  1.05  0.99  1.13  1.05  0.90
veal     1.08  1.09  1.00  0.93  0.89  0.96  0.98  1.02  1.02  1.01  0.98  1.03

beef     steady all year  peaks: -                lows: -
chicken  steady all year  peaks: -                lows: -
lamb     seasonal         peaks: Mar, A

In [6]:
WEATHER_YEARS = ("2015-01-01", "2024-12-31")
pasture_rows = []
for region, (lat, lon) in seasons.PASTURE_PLACES.items():
    scores = None
    if abs(lat) >= seasons.TROPICS_LATITUDE:            # no weather needed between the tropics
        daily = seasons.fetch_daily_weather(lat, lon, *WEATHER_YEARS, folder=WEATHER_DIR, refresh=REFRESH_DOWNLOADS)
        scores = seasons.grass_scores(seasons.monthly_weather(daily))
    season = seasons.pasture_season(lat, scores)
    pasture_rows.append({"region": region, "latitude": lat, "longitude": lon, **season,
                         "scores": [] if scores is None else scores["score"].round(2).tolist(),
                         "source": "Open-Meteo ERA5 2015-2024" if scores is not None else "tropics rule"})
    grass = ", ".join(MONTH_NAMES[m - 1] for m in season["grass_months"]) or "-"
    print(f"{region:36} {season['basis']:17} {grass}")

pasture_seasons = pd.DataFrame(pasture_rows)
pasture_seasons.to_csv("data/processed/pasture_seasons.csv", index=False)

Texas Gulf Coast (Houston)           year-round grass  Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec


US Great Plains (Nebraska)           seasonal          Apr, May, Jun, Jul, Aug, Sep, Oct


Western Canada (Alberta)             seasonal          May, Jun, Jul, Aug, Sep


Ireland (Cork)                       year-round grass  Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec


Central Spain (Madrid)               seasonal          Mar, Apr, May, Oct, Nov, Dec


Mongolia (Ulaanbaatar)               seasonal          Jul, Aug, Sep


Argentina pampas (Buenos Aires)      year-round grass  Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec


New Zealand (Waikato)                year-round grass  Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec


Southeast Australia (Wagga Wagga)    seasonal          Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec


US desert (Phoenix)                  no grass season   -
East Africa highlands (Nairobi)      tropics           -


In [7]:
# Check: known grazing months (approximate, from general agronomy; the team can refine them)
GRAZING_REFERENCE = {
    "Texas Gulf Coast (Houston)": range(1, 13),         # warm-season grass in summer, ryegrass in winter
    "US Great Plains (Nebraska)": range(4, 11),
    "Western Canada (Alberta)": range(5, 10),
    "Ireland (Cork)": range(2, 12),
    "Central Spain (Madrid)": [3, 4, 5, 10, 11, 12],    # dry summer, cool winter
    "Mongolia (Ulaanbaatar)": range(5, 10),
    "Argentina pampas (Buenos Aires)": range(1, 13),
    "New Zealand (Waikato)": range(1, 13),
    "Southeast Australia (Wagga Wagga)": range(4, 12),  # autumn break to late spring
    "US desert (Phoenix)": [],
}
check_rows = []
for row in pasture_seasons.itertuples():
    if row.region in GRAZING_REFERENCE:
        known, found = set(GRAZING_REFERENCE[row.region]), set(row.grass_months)
        check_rows.append({"region": row.region, "months_off": len(known ^ found),
                           "missed": sorted(known - found), "extra": sorted(found - known)})
pasture_check = pd.DataFrame(check_rows)
print(pasture_check.to_string(index=False))
print(f"\nMonths off: {pasture_check['months_off'].sum()} of {12 * len(pasture_check)}")

                           region  months_off missed extra
       Texas Gulf Coast (Houston)           0     []    []
       US Great Plains (Nebraska)           0     []    []
         Western Canada (Alberta)           0     []    []
                   Ireland (Cork)           1     []  [12]
           Central Spain (Madrid)           0     []    []
           Mongolia (Ulaanbaatar)           2 [5, 6]    []
  Argentina pampas (Buenos Aires)           0     []    []
            New Zealand (Waikato)           0     []    []
Southeast Australia (Wagga Wagga)           1     []  [12]
              US desert (Phoenix)           0     []    []

Months off: 4 of 120


---

## 4. Impact Class per Recipe

Without amounts, a recipe's class comes from its highest-impact main ingredient: **high** (beef, lamb,
cheese, prawns from farms), **medium** (pork, poultry, eggs, rice), **low** (legumes, grains, most
vegetables and fruit). Thresholds in kg CO2e per kg are a team decision; they must come from the source
table, not from guesses.

In [8]:
IMPACT_CLASSES = {"high": 20.0, "medium": 4.0}   # kg CO2e per kg at or above; placeholder until the team decides


def recipe_impact(recipes: pd.DataFrame, footprints: pd.DataFrame) -> pd.DataFrame:
    """recipe_id, impact_class, driver (the ingredient that sets the class)."""
    raise NotImplementedError("sketch: impact class")

---

## 5. Team Decisions and Next Steps

1. **Sources:** Our World in Data first (small, CC BY); AGRIBALYSE for prepared foods; confirm the USDA
   seasonal guide's license or pick another calendar.
2. **Impact thresholds** for high / medium / low.
3. **Regions** for the seasonal calendar, following notebook 03's pilot.
4. **Wording for the agent:** "a lower-impact choice" and "usually in season", never exact emissions per dish.

**Build order:** footprints (section 2), impact class (section 4), then seasons (section 3) once a calendar
source is confirmed.

---

## Plan status

In [9]:
PLAN = pd.DataFrame([
    ("Plan, sources, output tables", "written; licenses read 2026-10-05"),
    ("2 Footprint per ingredient", "stub - Our World in Data CSV (CC BY) ready to download"),
    ("3 What is in season", "stub - sources checked 2026-10-09; Ontario and Open-Meteo tested"),
    ("3b Meat: supply and pasture season", "built - USDA ERS and Open-Meteo, no keys"),
    ("4 Impact class per recipe", "stub - needs team thresholds"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))

                              step                                                           status
      Plan, sources, output tables                                written; licenses read 2026-10-05
        2 Footprint per ingredient           stub - Our World in Data CSV (CC BY) ready to download
               3 What is in season stub - sources checked 2026-10-09; Ontario and Open-Meteo tested
3b Meat: supply and pasture season                         built - USDA ERS and Open-Meteo, no keys
         4 Impact class per recipe                                     stub - needs team thresholds
